<a href="https://colab.research.google.com/github/MalakMostafa-16003651/nyc-housing-transportation-analysis/blob/main/Person5_Research_Questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Person 5 (Kenzy) — Visualizations, Map & Story
**RQ9 — Where in NYC do high affordable-housing production and high serious-crash levels overlap?** *(map)*
**RQ10 — Do years with higher affordable-housing production also have higher or lower severe injuries?** *(line chart + change scatter)*

**Inputs (from the repository root):**
- `df_integrated.csv`: Person 3's ZIP × year table (housing + SIRTA crashes, matched ZIP-years only)
- `df_housing_clean.csv`: Person 1's cleaned housing data (used for building coordinates and cleaned-snapshot yearly totals)
- `df_transport_clean.csv`: Person 2's cleaned SIRTA crash data (used for crash coordinates and cleaned-snapshot yearly totals)

**Datasets:** [Affordable Housing Production by Building (hg8x-zxpr)](https://data.cityofnewyork.us/d/hg8x-zxpr) · [SIRTA serious crashes (xeqp-qz8h)](https://data.cityofnewyork.us/d/xeqp-qz8h)

### Setup and shared chart style
Every chart in my section uses Plotly (the same library as the Dash website), so they can be reused there without redrawing. Colours are the website's shared palette from `app/charts/style.py`: **blue `#2a6fdb` = affordable housing**, **red `#c43d5a` = crashes/injuries**, and the same five borough colours, so the notebook and website look identical. `style_fig()` gives every figure the same title position, font, legend and source note.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from IPython.display import display

# Team colours: same meaning -> same colour in every chart
# Same values as app/charts/style.py (the website), so notebook and website match
COLOR_HOUSING = "#2a6fdb"   # blue   = affordable housing
COLOR_CRASH   = "#c43d5a"   # red    = serious crashes / injuries
COLOR_BOTH    = "#6a3d9a"   # purple = high on both
COLOR_OTHER   = "#c9ced3"   # grey   = context
BOROUGH_COLORS = {"Manhattan": "#2a6fdb", "Brooklyn": "#e0702b", "Queens": "#2f9e6e",
                  "Bronx": "#c43d5a", "Staten Island": "#7d5bbe"}
SOURCE_NOTE = "Source: NYC Open Data — Affordable Housing Production by Building (hg8x-zxpr); SIRTA crashes (xeqp-qz8h)"

def style_fig(fig, title, x_title=None, y_title=None, height=520, source=True):
    """Apply the team's common look to any Plotly figure."""
    fig.update_layout(
        template="plotly_white",
        title=dict(text=title, x=0.01, xanchor="left", font=dict(size=18)),
        font=dict(family="Arial", size=13, color="#222"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=60, r=30, t=95, b=80 if source else 50),
        height=height,
        hoverlabel=dict(bgcolor="white", font_size=12),
    )
    if x_title is not None: fig.update_xaxes(title_text=x_title)
    if y_title is not None: fig.update_yaxes(title_text=y_title)
    if source:
        fig.add_annotation(text=SOURCE_NOTE, xref="paper", yref="paper", x=0, y=-0.17,
                           showarrow=False, xanchor="left", font=dict(size=10, color="#777"))
    return fig

print("Style ready")

Style ready


### Load the team's data
I always reload the three CSVs from the repository root (like Person 1's notebook) so I never use a stale DataFrame. In Colab, where the folder starts empty, the cell opens an upload button for any missing file instead of crashing. ZIP codes are stored as 5-character text so they join and display the same way everywhere (e.g. `10456`, not `10456.0`).

In [2]:
from pathlib import Path

NEEDED = ["df_integrated.csv", "df_housing_clean.csv", "df_transport_clean.csv"]
missing = [f for f in NEEDED if not Path(f).is_file()]

if missing:
    try:
        from google.colab import files          # only exists in Colab
        print("Please upload:", ", ".join(missing))
        files.upload()                           # opens a file picker; select all missing CSVs
    except ImportError:
        raise FileNotFoundError(f"Missing {missing} — run from the repository root.")

still_missing = [f for f in NEEDED if not Path(f).is_file()]
if still_missing:
    raise FileNotFoundError(f"Still missing {still_missing}. Upload them with the exact same names.")

df_integrated      = pd.read_csv("df_integrated.csv")
df_housing_clean   = pd.read_csv("df_housing_clean.csv", low_memory=False)
df_transport_clean = pd.read_csv("df_transport_clean.csv", low_memory=False)

def zip_text(s):
    """Numeric/float ZIPs -> '10456' strings; missing stays missing."""
    return pd.to_numeric(s, errors="coerce").astype("Int64").astype("string").str.zfill(5)

for d in (df_integrated, df_housing_clean, df_transport_clean):
    d["zip_code"] = zip_text(d["zip_code"])
    d["year"] = pd.to_numeric(d["year"], errors="coerce").astype("Int64")

print("df_integrated:     ", df_integrated.shape)
print("df_housing_clean:  ", df_housing_clean.shape)
print("df_transport_clean:", df_transport_clean.shape)
print("SIRTA crash dates: ", df_transport_clean["crash_date"].min(), "to", df_transport_clean["crash_date"].max())

df_integrated:      (484, 11)
df_housing_clean:   (9551, 42)
df_transport_clean: (8072, 26)
SIRTA crash dates:  2022-10-01 to 2026-06-30


## RQ9 — Where do high affordable housing and serious-crash counts overlap?
Sum units and recorded serious crashes across available matched ZIP-years. Classify a ZIP as high when its total is at or above the 75th percentile of the 143-ZIP matched sample. Coverage varies across ZIPs and years; this is not an equal-exposure or citywide risk comparison. Housing uses project-start year; crash dates in the cleaned sample cover October 2022–June 2026. The raw source starts in January 2022, but many source records lack ZIPs.


In [3]:
zip_totals = (df_integrated.groupby("zip_code", as_index=False)
              .agg(borough=("borough", lambda s: s.mode().iat[0]),
                   affordable_units=("affordable_units", "sum"),
                   serious_crashes=("serious_crash_count", "sum"),
                   severe_injuries=("severe_injuries", "sum"),
                   fatalities=("fatalities", "sum"),
                   years_matched=("year", "nunique")))

HIGH_Q = 0.75
units_cut = zip_totals["affordable_units"].quantile(HIGH_Q)
crash_cut = zip_totals["serious_crashes"].quantile(HIGH_Q)
high_housing = zip_totals["affordable_units"].ge(units_cut) & zip_totals["affordable_units"].gt(0)
high_crashes = zip_totals["serious_crashes"].ge(crash_cut) & zip_totals["serious_crashes"].gt(0)

GROUP_ORDER = ["High on both", "High housing only", "High crashes only", "Neither"]
zip_totals["group"] = np.select([high_housing & high_crashes, high_housing, high_crashes],
                                GROUP_ORDER[:3], default="Neither")

print(f"{len(zip_totals)} ZIP codes")
print(f"'High housing' = top 25% = at least {units_cut:,.0f} affordable units")
print(f"'High crashes' = top 25% = at least {crash_cut:,.0f} serious crashes")
display(zip_totals["group"].value_counts().reindex(GROUP_ORDER).to_frame("ZIP codes"))

143 ZIP codes
'High housing' = top 25% = at least 1,172 affordable units
'High crashes' = top 25% = at least 65 serious crashes


,ZIP codes
group,
High on both,23
High housing only,13
High crashes only,15
Neither,92


### Map locations and coverage
Each position is the mean of available building and crash coordinates within the NYC bounding box. These are activity centroids, not official ZIP boundaries, and use all available source years for stable positioning. Missing or out-of-bounds coordinates are excluded only from positions. Raw point overlays below are restricted to matched ZIP-years, matching the totals; no unmapped event is subtracted from the counts.


In [4]:
def nyc_points(df, lat="latitude", lon="longitude"):
    pts = df[["zip_code", "year", lat, lon]].rename(columns={lat: "lat", lon: "lon"}).dropna()
    inside = pts["lat"].between(40.49, 40.92) & pts["lon"].between(-74.27, -73.68)
    return pts[inside]

all_housing_pts = nyc_points(df_housing_clean)
all_crash_pts = nyc_points(df_transport_clean)
centroids = pd.concat([all_housing_pts, all_crash_pts]).groupby("zip_code")[["lat", "lon"]].mean()
keys = df_integrated[["zip_code", "year"]]
housing_pts = all_housing_pts.merge(keys, on=["zip_code", "year"], validate="many_to_one")
crash_pts = all_crash_pts.merge(keys, on=["zip_code", "year"], validate="many_to_one")
zip_map = zip_totals.join(centroids, on="zip_code")
print("ZIPs without coordinates:", zip_map["lat"].isna().sum())
zip_map = zip_map.dropna(subset=["lat", "lon"])


ZIPs without coordinates: 0


### RQ9 map: housing and crash hotspots
- **Circles = ZIP codes.** Colour = group, size = affordable units. Hover over a circle for its numbers.
- Click **"Individual serious crashes"** or **"Affordable housing buildings"** in the legend to overlay the raw points from both datasets.
- Grey "Neither" ZIPs are drawn first so the coloured hotspots sit on top.

In [5]:
GROUP_COLORS = {"High on both": COLOR_BOTH, "High housing only": COLOR_HOUSING,
                "High crashes only": COLOR_CRASH, "Neither": COLOR_OTHER}
max_units = zip_map["affordable_units"].max()

fig_map = go.Figure()

# Optional raw layers (hidden until clicked in the legend)
fig_map.add_trace(go.Scattermap(
    lat=crash_pts["lat"], lon=crash_pts["lon"], mode="markers",
    marker=dict(size=4, color=COLOR_CRASH, opacity=0.35),
    name="Individual serious crashes", hoverinfo="skip", visible="legendonly"))
fig_map.add_trace(go.Scattermap(
    lat=housing_pts["lat"], lon=housing_pts["lon"], mode="markers",
    marker=dict(size=4, color=COLOR_HOUSING, opacity=0.35),
    name="Affordable housing buildings", hoverinfo="skip", visible="legendonly"))

# One layer per group, "Neither" first so hotspots are on top
for group in reversed(GROUP_ORDER):
    g = zip_map[zip_map["group"] == group]
    fig_map.add_trace(go.Scattermap(
        lat=g["lat"], lon=g["lon"], mode="markers", name=f"{group} ({len(g)})",
        marker=dict(size=8 + 30 * np.sqrt(g["affordable_units"] / max_units),   # sqrt: area ~ units
                    color=GROUP_COLORS[group], opacity=0.85),
        customdata=g[["zip_code", "borough", "affordable_units", "serious_crashes", "severe_injuries", "fatalities"]],
        hovertemplate=("<b>ZIP %{customdata[0]}</b> — %{customdata[1]}<br>"
                       "Affordable units: %{customdata[2]:,.0f}<br>"
                       "Serious crashes: %{customdata[3]:,.0f}<br>"
                       "Severe injuries: %{customdata[4]:,.0f}<br>"
                       "Fatalities: %{customdata[5]:,.0f}<extra></extra>")))

style_fig(fig_map, "Where affordable housing and serious crashes overlap — NYC ZIP codes, 2022–2026 (partial crash boundary years)",
          height=680, source=False)
fig_map.update_layout(map=dict(style="carto-positron", center=dict(lat=40.71, lon=-73.93), zoom=9.4),
                      margin=dict(l=10, r=10, t=95, b=10),
                      legend=dict(title=dict(text="Top 25% of ZIPs on…  (circle size = affordable units)  ")))
fig_map.show()

### Which ZIPs overlap, and how much do they account for?
This table backs up the map with exact numbers and measures how concentrated the problem is.

In [6]:
overlap = (zip_totals[zip_totals["group"] == "High on both"]
           .sort_values("serious_crashes", ascending=False)
           .drop(columns="group").reset_index(drop=True))

share_zips   = len(overlap) / len(zip_totals)
share_units  = overlap["affordable_units"].sum() / zip_totals["affordable_units"].sum()
share_crash  = overlap["serious_crashes"].sum() / zip_totals["serious_crashes"].sum()
share_deaths = overlap["fatalities"].sum() / zip_totals["fatalities"].sum()

print(f"{len(overlap)} of {len(zip_totals)} ZIPs ({share_zips:.0%}) are high on BOTH measures.")
print("By borough:", overlap["borough"].value_counts().to_dict())
print(f"They hold {share_units:.0%} of affordable units, {share_crash:.0%} of serious crashes "
      f"and {share_deaths:.0%} of crash deaths.")
rho, p = stats.spearmanr(zip_totals["affordable_units"], zip_totals["serious_crashes"])
print(f"Spearman ρ across all ZIPs = {rho:.2f} (p = {p:.1e}) — matches Person 1's RQ1.")
display(overlap)

23 of 143 ZIPs (16%) are high on BOTH measures.
By borough: {'Bronx': 11, 'Brooklyn': 8, 'Queens': 2, 'Manhattan': 2}
They hold 44% of affordable units, 34% of serious crashes and 29% of crash deaths.
Spearman ρ across all ZIPs = 0.64 (p = 3.8e-18) — matches Person 1's RQ1.


,zip_code,borough,affordable_units,serious_crashes,severe_injuries,fatalities,years_matched
0,11207,Brooklyn,2267.0,128.0,125.0,9.0,5
1,11212,Brooklyn,2698.0,119.0,115.0,8.0,5
2,11203,Brooklyn,1349.0,118.0,123.0,6.0,5
3,10456,Bronx,2968.0,109.0,112.0,2.0,5
4,11206,Brooklyn,1405.0,101.0,92.0,9.0,5
5,10467,Bronx,2402.0,100.0,102.0,8.0,5
6,11101,Queens,1771.0,98.0,94.0,6.0,5
7,10457,Bronx,3419.0,95.0,104.0,3.0,5
8,11208,Brooklyn,3578.0,90.0,98.0,5.0,5
9,10468,Bronx,2577.0,88.0,87.0,7.0,5


### RQ9 — Finding
23 of 143 matched ZIPs are high on both measures. They contain 44% of matched affordable units, 34% of serious crashes and 29% of fatalities. Most qualifying ZIPs are in the Bronx (11) and Brooklyn (8), with two each in Queens and Manhattan. Among qualifying ZIPs, 11207 and 11212 have the highest serious-crash counts. These are counts in the matched sample, rather than rankings of personal danger or evidence of a housing effect. The thresholds and exact totals are printed above.


## RQ10 — Housing production and severe injuries over time
The main trend holds ZIP coverage fixed: retain the 72 ZIPs represented in all three full calendar-year windows 2023–2025, then index each series to 2023 = 100. Boundary years are excluded. Missing-ZIP source exclusions and SIRTA reporting scope still apply; these are not citywide totals.

A second descriptive chart compares within-ZIP changes across 2023→2024 and 2024→2025. Differencing removes fixed ZIP attributes, but does not control time-varying confounding, reporting or project-start/occupancy timing. ZIPs can occur in both pairs, so naive independent-observation p-values are omitted. These plots cannot establish that housing causes, prevents, or has no effect on injuries.


In [7]:
FULL_YEARS = [2023, 2024, 2025]
panel = df_integrated[df_integrated["year"].isin(FULL_YEARS)].copy()
complete_zips = panel.groupby("zip_code")["year"].nunique()
complete_zips = complete_zips[complete_zips.eq(len(FULL_YEARS))].index
panel = panel[panel.zip_code.isin(complete_zips)]
yearly = panel.groupby("year")[["affordable_units", "severe_injuries", "serious_crash_count", "fatalities"]].sum()
yearly = yearly.rename(columns={"serious_crash_count": "serious_crashes"})
yearly.index = yearly.index.astype(int)
indexed = (yearly / yearly.loc[2023] * 100).round(1)
print("Balanced ZIPs:", len(complete_zips))
display(yearly.astype(int))
print("Change2023→2025:")
print((yearly.loc[2025] / yearly.loc[2023] - 1).map("{:+.1%}".format).to_string())


Balanced ZIPs: 72


,affordable_units,severe_injuries,serious_crashes,fatalities
year,,,,
2023,25494,1177,1173,69
2024,23881,1272,1254,86
2025,21562,1453,1403,82


Change2023→2025:
affordable_units    -15.4%
severe_injuries     +23.4%
serious_crashes     +19.6%
fatalities          +18.8%


### RQ10 chart A: balanced ZIP panel (2023 = 100)
Each series includes the same 72 ZIPs in each year. Labels show actual counts. A line above 100 indicates a count above that series' 2023 baseline, rather than greater personal risk.


In [8]:
fig_trend = go.Figure()
# Housing labels go below the line and injury labels above, so they never overlap
series = [("affordable_units", "Affordable units started", COLOR_HOUSING, "bottom center"),
          ("severe_injuries",  "Severe crash injuries",    COLOR_CRASH,   "top center")]
for col, label, color, position in series:
    fig_trend.add_trace(go.Scatter(
        x=indexed.index, y=indexed[col], name=label, mode="lines+markers+text",
        line=dict(color=color, width=3), marker=dict(size=10),
        text=[f"{v:,.0f}" for v in yearly[col]], textposition=position,
        customdata=yearly[col],
        hovertemplate=f"{label}<br>%{{x}}: %{{customdata:,.0f}} (index %{{y:.1f}})<extra></extra>"))

fig_trend.add_hline(y=100, line_dash="dot", line_color="#999")
style_fig(fig_trend, "Housing production and severe injuries — balanced 72-ZIP panel, 2023–2025",
          x_title="Year", y_title="Index (2023 = 100)")
fig_trend.update_xaxes(dtick=1)

fig_trend.add_annotation(text="Labels show actual yearly totals. 2022 and 2026 are excluded (partial years).",
                         xref="paper", yref="paper", x=0, y=1.0, showarrow=False, xanchor="left",
                         font=dict(size=11, color="#555"))
fig_trend.show()

### RQ10 chart B: within-ZIP changes
Each dot is a ZIP in one adjacent-year pair; both ZIP-years must be matched. Positive horizontal values mean increased affordable-unit counts; positive vertical values mean increased severe injuries. Repeated ZIPs and unmeasured changes limit inference.


In [9]:
wide = (df_integrated[df_integrated["year"].isin(FULL_YEARS)]
        .pivot_table(index=["zip_code", "borough"], columns="year",
                     values=["affordable_units", "severe_injuries"]))

pairs = []
for a, b in [(2023, 2024), (2024, 2025)]:
    pair = pd.DataFrame({
        "unit_change":   wide[("affordable_units", b)] - wide[("affordable_units", a)],
        "injury_change": wide[("severe_injuries", b)]  - wide[("severe_injuries", a)],
    }).dropna()                                   # ZIP must be matched in both years
    pair["years"] = f"{a}→{b}"
    pairs.append(pair)
changes = pd.concat(pairs).reset_index()

rho_c = changes["unit_change"].corr(changes["injury_change"], method="spearman")
r_c = changes["unit_change"].corr(changes["injury_change"])

fig_change = px.scatter(
    changes, x="unit_change", y="injury_change", color="years",
    color_discrete_sequence=["#93b7ee", "#1d4fa0"],   # light/dark housing blue = earlier/later pair
    hover_data={"zip_code": True, "borough": True, "years": True},
    labels={"unit_change": "Change in affordable units vs previous year",
            "injury_change": "Change in severe injuries vs previous year",
            "zip_code": "ZIP", "borough": "Borough", "years": "Years"})
# Straight-line fit over all dots (numpy, so no extra package is needed)
slope, intercept = np.polyfit(changes["unit_change"], changes["injury_change"], 1)
x_line = np.array([changes["unit_change"].min(), changes["unit_change"].max()])
fig_change.add_trace(go.Scatter(x=x_line, y=slope * x_line + intercept, mode="lines",
                                name="Trend (all dots)", line=dict(color="#444", dash="dash")))
fig_change.add_hline(y=0, line_color="#bbb"); fig_change.add_vline(x=0, line_color="#bbb")
fig_change.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color="white")),
                         selector=dict(mode="markers"))
style_fig(fig_change,
          f"Within-ZIP changes in housing and severe injuries (descriptive ρ={rho_c:.2f};n={len(changes)})",
          height=560)
fig_change.update_xaxes(tickformat=",", zeroline=False); fig_change.update_yaxes(zeroline=False)
fig_change.show()

print(f"Descriptive Spearman ρ={rho_c:+.2f}; Pearson r={r_c:+.2f}; n={len(changes)} ZIP-year changes; repeated ZIPs, no independent-sample p-value")
changes["housing_change"] = pd.cut(changes["unit_change"], [-np.inf, -100, 100, np.inf],
                                   labels=["Fell by >100 units", "Roughly flat (±100)", "Rose by >100 units"])
display(changes.groupby("housing_change", observed=True)["injury_change"]
        .agg(zip_years="size", avg_injury_change="mean", median_injury_change="median").round(2))

Descriptive Spearman ρ=-0.09; Pearson r=-0.13; n=167 ZIP-year changes; repeated ZIPs, no independent-sample p-value


,zip_years,avg_injury_change,median_injury_change
housing_change,,,
Fell by >100 units,56,2.21,3.0
Roughly flat (±100),62,1.71,1.5
Rose by >100 units,49,1.55,1.0


### RQ10 — Finding and interpretation
Read the balanced-panel table and percentage changes above for the exact trend. The within-ZIP change association is weakly negative (descriptive Spearman ρ approximately −0.09 across 167 changes). This does not establish an absence of effect: only three calendar-year windows are available, project starts precede occupancy, and reporting, traffic and other confounders are uncontrolled. Cross-sectional overlap and temporal changes answer different questions.


### Key numbers for the story
This cell reprints every number used in the findings and the story, so if the data is updated we can rerun it and check that the text above still matches the output.

In [10]:
# Key numbers for the story — rerun after any data change and update the markdown
print("=== RQ9 ===")
print(f"{len(overlap)}/{len(zip_totals)} ZIPs high on both | boroughs {overlap['borough'].value_counts().to_dict()}")
print(f"share of units {share_units:.0%}, crashes {share_crash:.0%}, deaths {share_deaths:.0%}")
print("Top 5:", ", ".join(f"{z} ({b})" for z, b in overlap[["zip_code", "borough"]].head(5).values))
print("\n=== RQ10 ===")
chg = yearly.loc[2025] / yearly.loc[2023] - 1
print(f"2023→2025: units {chg['affordable_units']:+.1%}, severe injuries {chg['severe_injuries']:+.1%}, deaths {chg['fatalities']:+.1%}")
print(f"Within-ZIP change correlation: descriptive ρ={rho_c:+.2f};n={len(changes)}; repeated ZIPs")

=== RQ9 ===
23/143 ZIPs high on both | boroughs {'Bronx': 11, 'Brooklyn': 8, 'Queens': 2, 'Manhattan': 2}
share of units 44%, crashes 34%, deaths 29%
Top 5: 11207 (Brooklyn), 11212 (Brooklyn), 11203 (Brooklyn), 10456 (Bronx), 11206 (Brooklyn)

=== RQ10 ===
2023→2025: units -15.4%, severe injuries +23.4%, deaths +18.8%
Within-ZIP change correlation: descriptive ρ=-0.09;n=167; repeated ZIPs


### Export for the website
Saves the three figures as standalone HTML and the tables behind them. It also writes **`data/zip_centroids.csv`**, the ZIP positions the website's RQ9 map (`app/charts/rq9_rq10.py`) reads, because `df_integrated` has no coordinates.

In [11]:
import os
os.makedirs("outputs", exist_ok=True)
for name, fig in {"rq9_overlap_map": fig_map, "rq10_trend": fig_trend, "rq10_zip_changes": fig_change}.items():
    fig.write_html(f"outputs/{name}.html", include_plotlyjs="cdn")
zip_map.to_csv("outputs/rq9_zip_overlap.csv", index=False)
yearly.to_csv("outputs/rq10_yearly_full_years.csv")
changes.to_csv("outputs/rq10_zip_changes.csv", index=False)
print("Saved:", sorted(os.listdir("outputs")))

# ZIP positions for the website map
os.makedirs("data", exist_ok=True)
centroids.round(6).reset_index().to_csv("data/zip_centroids.csv", index=False)
print("Saved data/zip_centroids.csv:", len(centroids), "ZIPs")

Saved: ['rq10_trend.html', 'rq10_yearly_full_years.csv', 'rq10_zip_changes.csv', 'rq10_zip_changes.html', 'rq9_overlap_map.html', 'rq9_zip_overlap.csv']
Saved data/zip_centroids.csv: 190 ZIPs


## Shared visualization rules applied in this audit
Housing is blue `#2a6fdb`, safety outcomes red `#c43d5a`, overlap purple, with consistent borough colours. Axes name the metric, unit and matched-sample scope. RQ1/RQ2 and RQ7/RQ8 exported PNGs are regenerated by their notebooks. The map shows activity centroids and exposes the classification threshold. Temporal comparisons exclude boundary years and RQ10 holds ZIP coverage fixed. Neither a ratio nor a correlation is a causal or exposure-adjusted safety measure.


# Project story: housing production and recorded serious crashes
Affordable housing production and SIRTA crash counts overlap in the selected NYC ZIP-years. Housing includes preservation and new construction; counts are dated by project start, rather than resident arrival.

Across 143 matched ZIPs, units and serious-crash totals have a positive rank association (RQ1 ρ≈0.64), and buildings and severe injuries have a positive association (RQ7 ρ≈0.73). Brooklyn contributes the largest shares of matched units and crashes. The 23 ZIPs above the 75th percentile on both measures include 11 in the Bronx and 8 in Brooklyn, accounting for 44% of matched units and 34% of crashes. This concentration can help identify places for further investigation.

RQ5 examines high activity at ZIP-year level; RQ6 compares crashes with observed affordable units. RQ8's crashes-per-project ratio is highest in Staten Island, but rests on only 15 summed project counts. These denominators are housing measures, rather than residents, trips or road exposure, and do not establish a borough's personal crash risk.

RQ3 shows changing matched-sample annual totals. RQ10 adds a balanced 72-ZIP panel for 2023–2025 and a descriptive within-ZIP change plot. Neither supports a causal conclusion about housing's effect on injuries or establishes that there is no effect. The displayed tables provide the current values. Lama's RQ11/RQ12 remain separate Person 6 deliverables.

**Limitations:** 13,754 raw SIRTA rows become 8,072 ZIP-known records, excluding 5,682 rows; this selection explains the cleaned subset's October 2022 start even though raw records begin in January 2022. Further integration retains 484 matched ZIP-years and excludes unmatched groups. Housing has 1,958 missing ZIPs and 29 unknown unit counts. Crash-coordinate gaps affect map positions but do not remove counted events. Activity centroids are approximate, ZIPs are coarse, and the same ZIP can be associated with different source boroughs. Three full calendar-year windows do not guarantee complete reporting. No population, traffic, road-length or occupancy denominator is available.

**Impact and next steps:**use the overlap as a screening tool to investigate street conditions near affordable housing, then add population, traffic exposure, validated ZIP geography, completion/occupancy dates and independent street-level evidence before recommending or evaluating interventions. Keep preservation separate from new construction when interpreting growth. The analysis does not support identifying already-dangerous neighbourhoods or claiming that new residents increased or reduced risk.


## Contribution — Person 5 (Kenzy)
| Area | Work |
|---|---|
| Research questions | RQ9 (overlap map), RQ10 (housing production vs severe injuries over time) |
| Visualizations | Interactive overlap map with toggleable raw crash/housing layers; indexed full-year trend; within-ZIP change scatter with correlation |
| Data work | ZIP centroids from building and crash coordinates; top-25 % overlap classification; full-year cleaned-snapshot totals; within-ZIP year-over-year changes |
| Consistency | Team colour and labelling rules, chart review checklist, fixes requested from teammates |
| Story | Question, data, findings (combining all team RQs), limitations and impact |
| Website | Wrote `app/charts/rq9_rq10.py` (RQ9 map + RQ10 chart, following the filters), `data/zip_centroids.csv`, and the story section text |